In [20]:
import pandas as pd
sheet=pd.read_csv(r"E:\Customer Trend analysis\Dataset\customer_shopping_behavior.csv")

In [21]:
sheet.shape

(3900, 18)

In [22]:
sheet.dtypes

Customer ID                 int64
Age                         int64
Gender                     object
Item Purchased             object
Category                   object
Purchase Amount (USD)       int64
Location                   object
Size                       object
Color                      object
Season                     object
Review Rating             float64
Subscription Status        object
Shipping Type              object
Discount Applied           object
Promo Code Used            object
Previous Purchases          int64
Payment Method             object
Frequency of Purchases     object
dtype: object

In [23]:
sheet.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [24]:
sheet.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [25]:
sheet['Review Rating']=sheet.groupby('Category')['Review Rating'].transform(lambda x:x.fillna(x.median()))

In [26]:
sheet.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [ ]:
sheet=sheet.rename(columns={'Purchase Amount (USD)':'purchase_amount'})
sheet.columns=sheet.columns.str.lower()
sheet.columns=sheet.columns.str.replace(' ','_')

In [28]:
sheet.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [32]:
print(sheet['age'].max())
print(sheet['age'].min())

70
18


In [36]:
labels=['Young Adult','Adult','Middle Aged','Senior']

sheet['age_group']=pd.qcut(sheet['age'],q=4,labels=labels)

In [37]:
sheet[['age','age_group']].head(5)

,age,age_group
0,55,Middle Aged
1,19,Young Adult
2,50,Middle Aged
3,21,Young Adult
4,45,Middle Aged


In [39]:
sheet['frequency_of_purchases'].unique()

array(['Fortnightly', 'Weekly', 'Annually', 'Quarterly', 'Bi-Weekly',
       'Monthly', 'Every 3 Months'], dtype=object)

In [40]:
frequency_mapping={
    'Fortnightly':14,
    'Weekly':7,
    'Bi-Weekly':14,
    'Monthly':30,
    'Quarterly':90,
    'Every 3 Months':90,
    'Annually':365,
}

sheet['purchase_frequency']=sheet['frequency_of_purchases'].map(frequency_mapping)

In [41]:
sheet[['purchase_frequency','frequency_of_purchases']].head(5)

,purchase_frequency,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually


In [ ]:
(sheet['discount_applied']==sheet['promo_code_used']).all()

#Since both the columns have same value we will drop one column

np.True_

In [46]:
sheet=sheet.drop('promo_code_used',axis=1)

In [47]:
sheet.head(5)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle Aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle Aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle Aged,365


In [48]:
sheet['payment_method'].unique()

array(['Venmo', 'Cash', 'Credit Card', 'PayPal', 'Bank Transfer',
       'Debit Card'], dtype=object)

In [49]:
payment={
    'Cash':1,
    'Credit Card':2,
    'Debit Card':3,
    'PayPal':4,
    'Venmo':5,
    'Bank Transfer':6
}

sheet['payment_num']=sheet['payment_method'].map(payment)

In [53]:
sheet[['payment_num','payment_method']].head(5)

,payment_num,payment_method
0,5,Venmo
1,1,Cash
2,2,Credit Card
3,4,PayPal
4,4,PayPal


In [ ]:
from sqlalchemy import create_engine

username="#######"
password="#######"
host="#######"
port="#####"
database="######"

engine=create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name="customer"

sheet.to_sql(table_name,engine,if_exists="replace",index=False)

print("Data Loaded")

Data Loaded
